In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In this notebook, I will recreate n-gram model,
which is just a list where you can navigate till (n-1) words
"I am" (2) words before and probability from speech corpus "Firdavs" - 0.8 "a" - 0.3 ... something like this
but I do not think that this is a good idea (it is like bag-of-words) but we will see.

In [2]:
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

--2025-09-22 16:37:44--  https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1 [following]
--2025-09-22 16:37:44--  https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc60e09d658f82e70d82f859fdfa.dl.dropboxusercontent.com/cd/0/inline/Cx3h5jW8AY7L1Q9eL7-WH7ywHvbyTg7E5yPP4-NRjyrxuiN2YuXQbLFfto6BYsxA7fn4fFMvGjTQiG4ZPmAW_pepYLydEKudcV7a81KcQ-j6OpqXh1HjyWHoRpZIQbzvIKY/file?dl=1# [following]
--2025-09-22 16:37:44--  https://uc60e09d658f82e70d82f859fdfa.dl.dropboxuse

,author,day,id,link,month,summary,tag,title,year
18626,"[{'name': 'Francesca Giardini'}, {'name': 'Wal...",21,1106.4218v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,"The study of opinions, their formation and cha...","[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Rooting opinions in the minds: a cognitive mod...,2011
19975,"[{'name': 'Paul E. Lehner'}, {'name': 'Azar Sa...",20,1303.5729v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,A series of monte carlo studies were performed...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Reasoning under Uncertainty: Some Monte Carlo ...,2013
19291,[{'name': 'Benjamin N. Grosof'}],27,1304.3087v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,We start by defining an approach to non-monoto...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Non-Monotonicity in Probabilistic Reasoning,2013
26533,"[{'name': 'Mina Nouredanesh'}, {'name': 'Andre...",2,1611.00684v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,"In this paper, a method to detect environmenta...","[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Wearable Vision Detection of Environmental Fal...,2016
17429,"[{'name': 'Yu-An Chung'}, {'name': 'James Glas...",5,1711.01515v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,"In this paper, we propose a novel deep neural ...","[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Learning Word Embeddings from Speech,2017


In [3]:
print(data["title"][0])
print(data["summary"][0])

Dual Recurrent Attention Units for Visual Question Answering
We propose an architecture for VQA which utilizes recurrent layers to
generate visual and textual attention. The memory characteristic of the
proposed recurrent attention units offers a rich joint embedding of visual and
textual features and enables the model to reason relations between several
parts of the image and question. Our single model outperforms the first place
winner on the VQA 1.0 dataset, performs within margin to the current
state-of-the-art ensemble model. We also experiment with replacing attention
mechanisms in other state-of-the-art models with our implementation and show
increased accuracy. In both cases, our recurrent attention mechanism improves
performance in tasks requiring sequential or relational reasoning on the VQA
dataset.


In [4]:
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

In [5]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer

from nltk.tokenize import WordPunctTokenizer
tk = WordPunctTokenizer()

geek = tk.tokenize("some text for tokenization, 123 , 1 2 3 23")


for idx, line in enumerate(lines):
    tok_line = tk.tokenize(line.strip())
    lines[idx] = " ".join(tok_line).lower()

print(lines[0])

dual recurrent attention units for visual question answering ; we propose an architecture for vqa which utilizes recurrent layers to generate visual and textual attention . the memory characteristic of the proposed recurrent attention units offers a rich joint embedding of visual and textual features and enables the model to reason relations between several parts of the image and question . our single model outperforms the first place winner on the vqa 1 . 0 dataset , performs within margin to the current state - of - the - art ensemble model . we also experiment with replacing attention mechanisms in other state - of - the - art models with our implementation and show increased accuracy . in both cases , our recurrent attention mechanism improves performance in tasks requiring sequential or relational reasoning on the vqa dataset .


In [6]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

In [7]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in lines:
      tokens = line.strip().split()+[EOS]
      tokens.append(EOS)

      for i in range(len(tokens)):
        if i < n - 1:
          prefix = [UNK] * (n - 1 - i) + tokens[:i]
        else:
          prefix = tokens[i - (n - 1):i]

        prefix = tuple(prefix)
        next_token = tokens[i]
        counts[prefix][next_token] += 1

    return counts

In [8]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

In [9]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)

        self.probs = defaultdict(Counter)
        for prefix, token_counts in counts.items():   # <-- используем prefix напрямую
            for token, cnt in token_counts.items():
                self.probs[prefix][token] += cnt

        # normalize into probabilities
        for prefix, counter in self.probs.items():
            total = float(sum(counter.values()))
            if total > 0:
                for token in list(counter.keys()):
                    counter[token] = counter[token] / total

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

In [10]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

In [11]:
lm = NGramLanguageModel(lines, n=3)

In [12]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    Return next token after prefix.

    :param lm: NGramLanguageModel instance
    :param prefix: string, space-separated prefix tokens
    :param temperature:
        - if temperature == 0: take the most likely token
        - else: sample according to P(token)^(1/temperature)
    """
    probs = lm.get_possible_next_tokens(prefix)
    if not probs:
        return "_UNK_"

    tokens = list(probs.keys())
    probabilities = np.array(list(probs.values()), dtype=np.float64)

    if temperature == 0:
        # deterministic: pick the highest probability token
        return tokens[np.argmax(probabilities)]

    # temperature scaling
    scaled_probs = probabilities ** (1.0 / temperature)
    scaled_probs /= scaled_probs.sum()  # normalize

    return np.random.choice(tokens, p=scaled_probs)

In [13]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000
print("Looks nice!")

Looks nice!


In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial neural network utilizes application - specific features , and performance of all samples in target recognition ( cer ). since the absence of reliable and at the resected lung surface is reduced from o ( n \ geq c \ subseteq ...$ which correspond to the features of pre - aspiration duration : a critique of scheduled events from irregularly sampled and noisy input data into a robust method for inferring the individualized causal effects , thus different initial conditions and the distances between examples and show that the network to spectral methods , focusing on predicting successful anti -


In [15]:
prefix = "the jabberwock , with eyes of flame , "

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

the jabberwock , with eyes of flame ,  _UNK_


In [16]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    total_logprob = 0.0
    total_tokens = 0

    for line in lines:
        tokens = line.strip().split()
        # add EOS marker to evaluate probability of ending
        tokens.append("_EOS_")

        prefix = ""
        for tok in tokens:
            # get distribution for next token given current prefix
            probs = lm.get_possible_next_tokens(prefix)

            # probability of the actual token (if unseen → very small)
            p_tok = probs.get(tok, 0.0)
            if p_tok <= 0.0:
                logp = min_logprob
            else:
                logp = np.log(p_tok)

            total_logprob += logp
            total_tokens += 1

            # update prefix (space-separated string)
            prefix = (prefix + " " + tok).strip()

    # average negative log-likelihood
    avg_neg_logprob = - total_logprob / total_tokens
    ppl = np.exp(avg_neg_logprob)
    return ppl

In [17]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
#assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249)) <- makes no sense if the datset it updfated bro

Perplexities: ppx1=321.436 ppx3=1.520 ppx10=1.184


In [18]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 1835.16731
N = 2, Perplexity = 85653987.28774
N = 3, Perplexity = 61999196259043346743296.00000


In [19]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))

In [20]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [21]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

KeyboardInterrupt: 

In [24]:
class KneserNeyLanguageModel:
    def __init__(self, lines, n, delta=1.0):
        """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
        self.n = n
        self.delta = float(delta)

        self.ngram_counts = {k: Counter() for k in range(1, n + 1)}
        self.vocab = set()

        # Build n-gram counts
        for line in lines:
            tokens = line.strip().split()
            tokens.append("_EOS_")
            self.vocab.update(tokens)
            L = len(tokens)
            for k in range(1, n + 1):
                if L < k:
                    continue
                for i in range(0, L - k + 1):
                    ng = tuple(tokens[i:i + k])
                    self.ngram_counts[k][ng] += 1

        self.prefix_counts = {m: Counter() for m in range(0, n)}
        self.continuations = {m: defaultdict(set) for m in range(0, n)}

        for m in range(1, n + 1):
            if m == 1:
                continue
            for ng, cnt in self.ngram_counts[m].items():
                prefix = ng[:-1]
                token = ng[-1]
                self.prefix_counts[m - 1][prefix] += cnt
                self.continuations[m - 1][prefix].add(token)

        total_unigrams = sum(self.ngram_counts[1].values())
        self.prefix_counts[0][()] = total_unigrams

        self.left_context_counts = Counter()
        if 2 in self.ngram_counts and len(self.ngram_counts[2]) > 0:
            for (u, w), cnt in self.ngram_counts[2].items():
                self.left_context_counts[w] += 1
            self.total_bigram_types = len(self.ngram_counts[2])
        else:
            self.total_bigram_types = 0

    def get_possible_next_tokens(self, prefix):
        probs = {}
        for tok in self.vocab:
            probs[tok] = self.get_next_token_prob(prefix, tok)
        return probs

    def get_next_token_prob(self, prefix, next_token):
        if prefix is None:
            prefix_tokens = []
        else:
            prefix_tokens = prefix.strip().split() if prefix.strip() != "" else []
        if len(prefix_tokens) > self.n - 1:
            prefix_tokens = prefix_tokens[-(self.n - 1):]
        history = tuple(prefix_tokens)

        def prob(history_tuple, word):
            m = len(history_tuple) + 1
            if m == 1:
                if self.total_bigram_types > 0:
                    return self.left_context_counts.get(word, 0) / float(self.total_bigram_types)
                total_unigrams = float(self.prefix_counts[0].get((), 0) or 1)
                return self.ngram_counts[1].get((word,), 0) / total_unigrams

            ng = history_tuple + (word,)
            c_hw = self.ngram_counts.get(m, Counter()).get(ng, 0)

            c_h = self.prefix_counts.get(m - 1, Counter()).get(history_tuple, 0)

            if c_h == 0:
                return prob(history_tuple[1:], word)

            D = self.delta
            first = max(c_hw - D, 0.0) / c_h
            cont_count = len(self.continuations.get(m - 1, {}).get(history_tuple, ()))
            lam = (D * cont_count) / c_h if c_h > 0 else 0.0

            lower = prob(history_tuple[1:], word)
            return first + lam * lower

        return prob(history, next_token)


In [25]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [26]:
delta = 0.75

for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=delta)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

KeyboardInterrupt: 